# **Underpriced Cars**

- Seperate Testing Environment of the market_model
- Training & Plotting in the market_model Notebook

**Active listings as of 2025.09.07**

In the best-case scenario, all active listings on the site would be predicted here. However, I don't have enough inactive listings for training yet, so I have to include some active listings in the training and validation aswell. This means those listings can't be tested / predicted here afterward, obviously.


---

In [1]:
import torch
import pandas as pd
from torch.utils.data import DataLoader

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import sys
sys.path.insert(0, '../')

from modeling.dataset import CarPriceDataset
from modeling.model import MLPCarPriceRegressionNet_V1, MLPCarPriceRegressionNet_V2
from modeling.test import test_model
from modeling.plots_and_metrics import find_undervalued_cars

from preprocessing.preprocess import PreProcessor

## **Testing Prep & Testing**

---

In [2]:
MODEL_NAME = 'market_model'
CAR_DETAILS_DATASET_TEST = 'car_details_155998_20250907.csv'
CAR_DATA_DATASET = 'car_data_155998_20250907.csv'

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
car_data = pd.read_csv(f'../data/raw/{CAR_DATA_DATASET}')
df_test = pd.read_csv(f'../workspace/test_datasets/{CAR_DETAILS_DATASET_TEST}')

preprocessor = PreProcessor()
preprocessor.load(f'../workspace/config/{MODEL_NAME}.pkl')

batch_size = 64
X_num_test, X_cat_test, y_test = preprocessor.transform(df_test, is_training=False, include_target=True)
test_dataset = CarPriceDataset(X_num_test, X_cat_test, y_test)
dataloader_test = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

model = MLPCarPriceRegressionNet_V2(f'../workspace/config/{MODEL_NAME}.pkl')
model.load_state_dict(torch.load(f'../workspace/models/{MODEL_NAME}.pt'))

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

results = test_model(model, dataloader_test)

## **Potential good deals**

Cars my model thinks are undervalued by more than 50% +

*There ought to be some "errors" obviously, For example the first car here if we open the link and read the listings description, its not actually listed at 4M HUF, but rather its a 4M HUF down payment with monthly installments + residual value, so adding it up alltogether comes down to about 24M HUF.*

In [3]:
display(find_undervalued_cars(results, df_test, car_data, undervalued_threshold=50))

actual_price,predicted_price,undervalued_percent,manufacturer,model,year,kilometers,kw,url
"3,989,999 Ft","25,091,272 Ft",+528.9%,AUDI,Q7,2022,"99,000 km",250,Car page
"6,000,000 Ft","13,287,929 Ft",+121.5%,PEUGEOT,208,2022,"43,300 km",100,Car page
"15,000,010 Ft","32,569,636 Ft",+117.1%,PORSCHE,PANAMERA,2016,"52,050 km",228,Car page
"13,994,993 Ft","27,250,556 Ft",+94.7%,MERCEDES-BENZ,GLC-OSZTÁLY GLC 350,2019,"142,331 km",190,Car page
"3,490,000 Ft","6,679,608 Ft",+91.4%,INFINITI,QX QX70,2017,"210,000 km",175,Car page
"8,190,000 Ft","15,314,074 Ft",+87.0%,OPEL,MOKKA,2023,"7,200 km",100,Car page
"8,490,003 Ft","15,826,405 Ft",+86.4%,RENAULT,CAPTUR,2025,"5,692 km",116,Car page
"2,490,000 Ft","4,491,766 Ft",+80.4%,BMW,3-AS SOROZAT 325,2012,"271,324 km",160,Car page
"17,999,016 Ft","31,897,492 Ft",+77.2%,MERCEDES-AMG,C-OSZTÁLY C 63,2019,"47,000 km",375,Car page
"2,999,998 Ft","5,204,828 Ft",+73.5%,AUDI,ALLROAD A4 ALLROAD,2012,"225,400 km",176,Car page
